# Why can hedging strategies carry more risk?

#
<!-- Directional market risk: if the hedge behaves differently from what you expected, losses can be larger than planned.

Leverage: many hedging strategies use leverage to amplify returns, and the same leverage amplifies losses.

Liquidity risk: in stressed markets, some hedges may be difficult to trade in time, especially when derivatives or illiquid assets are involved. -->


---
### Factor in Practice Part 3
# Build your own hedge fund: implementing a hedging strategy in Python

##### "I do not know what the future holds, but I know where I will be waiting for it." - Warren Buffett

### 🎬 @Director Harold
### 🏛 CUHK Financial Engineering Undergraduate Program
### 📈 On the path to a U.S. Master's in Financial Engineering (already admitted)
### 🌐 [Follow my Bilibili channel for quant learning content that is easy to follow and actually useful](https://space.bilibili.com/629573485)

🌟🌟🌟 Let's pull back the curtain on quantitative investing. #HaroldQuantChannel 🌟


---


Load data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Use a representative subset of S&P 500 stocks
sp500_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'BRK-B', 'JNJ', 'V', 'WMT',
    'JPM', 'PG', 'MA', 'UNH', 'HD', 'DIS', 'BAC', 'NVDA', 'ADBE', 'CRM',
    'NFLX', 'PYPL', 'INTC', 'VZ', 'T', 'MRK', 'PFE', 'ABT', 'TMO', 'NKE',
    'XOM', 'CVX', 'LLY', 'MCD', 'KO', 'PEP', 'ABBV', 'AVGO', 'COST', 'ACN',
    'TXN', 'DHR', 'NEE', 'UPS', 'PM', 'LIN', 'LOW', 'QCOM', 'SPGI', 'IBM',
    'BMY', 'AMGN', 'GS', 'BLK', 'CAT', 'GE', 'MMM', 'AXP', 'SCHW', 'BKNG'
]

start = '2005-01-01'
end = '2022-12-31'

monthly_data = []
for ticker in sp500_tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start, end=end, interval='1mo')
        hist = hist.reset_index()
        hist['stock'] = ticker
        hist['date'] = hist['Date'].dt.strftime('%Y-%m')
        hist['monthly_return'] = hist['Close'].pct_change()
        info = stock.info
        hist['market_cap'] = hist['Close'] * info.get('sharesOutstanding', 1e9)
        hist['value_in_thousand'] = hist['market_cap'] / 1000
        hist['next_month_return'] = hist['monthly_return'].shift(-1)
        monthly_data.append(hist[['stock', 'date', 'value_in_thousand', 'monthly_return', 'next_month_return']])
    except:
        pass

df_mkt = pd.concat(monthly_data, axis=0).dropna(subset=['monthly_return']).reset_index(drop=True)
df_mkt

In [ ]:
df = df_mkt
unique_dates = df['date'].unique()
unique_dates.sort()

In [ ]:
df = df_mkt
unique_dates = df['date'].unique()
unique_dates.sort()

---

Start with a simple hedging idea. Suppose we believe small-cap stocks are more attractive than large-cap stocks. Then we can buy the smallest 1% of stocks in the market and short the largest 1%. That gives us a hedged portfolio.

The beta of this portfolio should be close to zero, meaning its returns should be less dependent on the market. In principle, that lets us hedge away market beta risk and pursue an alpha strategy instead.

That is why this is also called a market-neutral strategy or a hedged strategy.

---


In [ ]:
long_percentile = 1
short_percentile = 99

strategy_returns = []

for date in unique_dates:

    df_date = df[df['date'] == date]

    long_threshold = np.percentile(df_date['value_in_thousand'], long_percentile)
    short_threshold = np.percentile(df_date['value_in_thousand'], short_percentile)

    long_stocks = df_date[df_date['value_in_thousand'] <= long_threshold]['stock']
    short_stocks = df_date[df_date['value_in_thousand'] >= short_threshold]['stock']

    df_long_stock = df_date[df_date['stock'].isin(long_stocks)]
    df_short_stock = df_date[df_date['stock'].isin(short_stocks)]

    # Key point here! We use next month's return to compute portfolio return,
    # since we build the portfolio based on this month's end market cap.
    long_return = df_long_stock['next_month_return'].mean()
    short_return = df_short_stock['next_month_return'].mean()

    strategy_return = long_return - short_return
    strategy_returns.append(strategy_return)


A visual illustration


In [ ]:
unique_dates_nochange = unique_dates.copy()
unique_dates = [datetime.strptime(date, "%Y-%m") for date in unique_dates]

In [ ]:
plt.style.use('seaborn-darkgrid')  
plt.figure(figsize=(10, 6)) 
plt.plot(unique_dates, strategy_returns, label='Strategy Returns', color='red', linewidth=2)
plt.title('Strategy Returns Over Time')  # Add title
plt.xlabel('Date')  # Add x-axis label
plt.ylabel('Returns')  # Add y-axis label
plt.legend()  # Show legend
plt.grid(True)  # Show grid
plt.show()
pd.DataFrame(strategy_returns).describe()

In [ ]:
cumulated_returns = np.cumsum(strategy_returns)
plt.style.use('seaborn-darkgrid')
plt.figure(figsize=(10, 6))
plt.plot(unique_dates, cumulated_returns, label='Strategy  the cumulative returns', color='red', linewidth=2)
plt.title('Strategy Returns Over Time', fontsize=20, weight='bold')
plt.xlabel('Date', fontsize=16)
plt.ylabel('Cumulative Returns', fontsize=16)
plt.grid(True)
plt.show()
cumulated_returns

---
# What else do we still need to think about?
## What problems might show up? Is this stock-selection rule too aggressive? Can we derive a conclusion that works across the full market instead of only at the extremes?

---


In [ ]:
# Initialize return lists for each portfolio
portfolio_returns = {f'portfolio_{i}': [] for i in range(1, 6)}

for date in unique_dates_nochange:

    df_date = df[df['date'] == date]
    
    # Check if df_date is empty
    if df_date.empty:
        # If empty, skip this date
        continue
    
    # Get quintile thresholds
    thresholds = np.percentile(df_date['value_in_thousand'], [20, 40, 60, 80, 100])
    
    # Build five portfolios
    for i in range(5):
        if i == 0:
            # First portfolio starts from minimum
            portfolio = df_date[df_date['value_in_thousand'] <= thresholds[i]]
        else:
            # Subsequent portfolios between two thresholds
            portfolio = df_date[(df_date['value_in_thousand'] > thresholds[i-1]) & (df_date['value_in_thousand'] <= thresholds[i])]
        
        # If portfolio is empty, continue to next
        if portfolio.empty:
            portfolio_return = 0
        else:
            # Here we still look at next month's return
            if not np.isnan(portfolio['next_month_return'].mean()):
                portfolio_return = portfolio['next_month_return'].mean()

        portfolio_returns[f'portfolio_{i+1}'].append(portfolio_return)


In [ ]:
# Hedge portfolio return is long portfolio 1 minus short portfolio 5
hedge_returns = []
for i in range(len(portfolio_returns['portfolio_1'])):
    hedge_return = portfolio_returns['portfolio_1'][i] - portfolio_returns['portfolio_5'][i]
    hedge_returns.append(hedge_return)

# Output average return for each portfolio
average_returns = {portfolio: np.mean(returns) for portfolio, returns in portfolio_returns.items()}
average_hedge_return = np.mean(hedge_returns)

# Print results
print("Average Returns by Portfolio:")
for portfolio, average_return in average_returns.items():
    print(f"{portfolio}: {average_return}")

print(f"Average Hedge Return: {average_hedge_return}")

In [ ]:
# Calculate the cumulative returns for each portfolio
cumulative_portfolio_returns = {portfolio: np.cumsum(returns) for portfolio, returns in portfolio_returns.items()}
# Calculate the cumulative hedge returns
cumulative_hedge_returns = np.cumsum(hedge_returns)
# Get the final cumulative return for each portfolio and the hedge strategy
final_cumulative_returns = {portfolio: returns[-1] if len(returns) > 0 else 0 for portfolio, returns in cumulative_portfolio_returns.items()}
final_cumulative_hedge_return = cumulative_hedge_returns[-1] if len(cumulative_hedge_returns) > 0 else 0
# Print the final cumulative returns by portfolio
print("Final Cumulative Returns by Portfolio:")
for portfolio, final_return in final_cumulative_returns.items():
    print(f"{portfolio}: {final_return}")
# Print the final cumulative hedge return
print(f"Final Cumulative Hedge Return: {final_cumulative_hedge_return}")


In [ ]:
# Calculate the cumulative returns for each portfolio
cumulative_portfolio_returns = {portfolio: np.cumsum(returns) for portfolio, returns in portfolio_returns.items()}

# Plotting the cumulative returns for each portfolio
plt.figure(figsize=(14, 7))  # Set the figure size

for portfolio, returns in cumulative_portfolio_returns.items():
    plt.plot(unique_dates, returns, label=portfolio)

plt.title('Cumulative Portfolio Returns Over Time')
plt.xlabel('Time Periods')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

---

# Factors...?

Harold, didn't you say that a factor can help determine whether a stock goes up or down?

---


Harold two-factor model


In [ ]:
SMB_monthly = pd.DataFrame(hedge_returns, unique_dates_nochange,columns=['SMB_monthly'])

pd.DataFrame(hedge_returns, unique_dates_nochange,columns=['SMB_monthly'])

In [ ]:
df_mkt

In [ ]:
MKT = df_mkt.groupby('date')['monthly_return'].mean()
MKT_monthly = pd.DataFrame(MKT)
MKT_monthly

Now we have monthly values for the MKT factor and the SMB factor.
### For any stock, its monthly return can be decomposed as R = A1 * MKT factor + A2 * SMB factor + alpha


In [ ]:
test_stock_code = 'JPM'  # JPMorgan Chase

test_stock = df_mkt[df_mkt['stock'] == test_stock_code]
test_stock

In [ ]:
X = pd.concat([SMB_monthly['SMB_monthly'], MKT_monthly['monthly_return']],axis= 1)
X

In [ ]:
# regression on next_month_return on SMB
import statsmodels.api as sm
from statsmodels import regression

test_stock.fillna(0, inplace=True)

# X = SMB_monthly['SMB_monthly'],MKT_monthly['monthly_return']
y = test_stock[['date','next_month_return']].set_index('date')

data_for_stock = test_stock['date'].unique()
data_for_stock.sort()
y = y.loc[data_for_stock]
X = X.loc[data_for_stock]

X = sm.add_constant(X)
model = regression.linear_model.OLS(y, X).fit()
alpha = model.params[0]
beta1 = model.params[1]
beta2 = model.params[2]
print('alpha: ' + str(alpha))
print('beta SMB: ' + str(beta1))
print('beta MKT: ' + str(beta2))
print(model.summary())

---

# Three factors, and then more...

---


In [ ]:
# Load PB ratio data via yfinance for S&P 500 stocks
pb_data = []
for ticker in sp500_tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start='2005-01-01', end='2022-12-31', interval='1mo')
        hist = hist.reset_index()
        hist['stock'] = ticker + '.US'
        info = stock.info
        name = info.get('shortName', ticker)
        hist['name'] = name
        hist['date'] = hist['Date']
        hist['return'] = hist['Close'].pct_change() * 100
        hist['PB'] = info.get('priceToBook', np.nan)
        hist['next_month_return'] = hist['return'].shift(-1)
        pb_data.append(hist[['stock', 'name', 'date', 'return', 'PB', 'next_month_return']])
    except:
        pass

df_pb = pd.concat(pb_data, axis=0).reset_index(drop=True)
df_pb

In [ ]:
df = df_pb.copy()
df['book_to_market'] = 1 / df['PB']
df

# TODO: build an investment portfolio using the book-to-market ratio


# How can we replicate the Fama-French three-factor model in the US market?

### Reference: Fama + French (1993)
